# Notebook 08 — Transformer Input Preprocessing Ablation

Train HeartBERT, ECG-PT, and HuBERT-ECG with model-specific
preprocessing (bandpass filter + normalisation) and compare
against the raw-signal baselines from Notebook 05.

**Experiments (LoRA only, r=8, alpha=16):**
| Model | Preprocessing |
|---|---|
| HeartBERT | Bandpass 0.5–40 Hz + per-signal z-score |
| ECG-PT | Bandpass 0.5–40 Hz only |
| HuBERT-ECG | Bandpass 0.5–40 Hz + per-lead z-score |

Results saved to `results/08_preprocessing_ablation/`.
Test-set evaluation compared against r=8 raw baselines from Notebook 05.

In [ ]:
import sys, os, warnings, json, gc
from pathlib import Path
sys.path.append('../')
warnings.filterwarnings('ignore')

import numpy as np
import torch
import wfdb
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from torch.utils.data import Dataset, DataLoader

from src.utils.config import CFG
from src.preprocessing.label_utils import load_all_labels, SUPERCLASSES
from src.evaluation.metrics import full_eval, compute_auc, compute_probs

DATA_PATH    = CFG['data']['path']
RESULTS_PATH = CFG['paths']['results']
HUBERT_SIZE  = CFG['model']['hubert_size']
OUT_DIR      = RESULTS_PATH + '08_preprocessing_ablation/'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'OutDir : {OUT_DIR}')

## 1. Motivation and Preprocessing Assumptions

All three Transformers were trained in Notebook 04/05 on **raw, unfiltered signals**
loaded directly from the PTB-XL WFDB files with no preprocessing. The preprocessing
ablation in Notebook 02–03 was conducted exclusively for FCN-Wang. This notebook
investigates whether model-specific preprocessing improves Transformer performance.

---

### HeartBERT — Bandpass + Z-score

HeartBERT encodes Lead II amplitudes by uniformly quantising the signal range into
32 bins and mapping each sample to a letter. The bin boundaries are set as
`np.linspace(signal.min(), signal.max(), 32)` — computed per signal.

**Problem with raw input:**
- Baseline wander (drift < 0.5 Hz) artificially inflates the amplitude range, pushing
  most of the cardiac signal into a narrow band of letters in the middle.
- Two patients with identical cardiac morphology but different baseline offsets or
  electrode impedances will receive completely different letter sequences.

**Why bandpass + z-score:**
- Bandpass (0.5–40 Hz) removes baseline wander and EMG noise.
- Z-score (`mean=0, std=1`) makes the bin boundaries amplitude-invariant:
  all signals map to the same dynamic range, so the 32 bins always represent
  the same physiological amplitude increments regardless of the patient.

---

### ECG-PT — Bandpass only

ECG-PT splits Lead II into 36-sample patches and normalises each patch internally:
`norm = (patch.mean() - lo) / (hi - lo + 1e-8)`. Amplitude is already handled
per-patch, so adding global z-score would be double-normalisation and would flatten
amplitude differences between patches that carry diagnostic information (e.g.
high-voltage QRS in HYP vs low-voltage T-waves in STTC).

**Why bandpass only:**
- Baseline wander shifts the mean of each patch and therefore shifts all token IDs
  systematically, breaking the patch tokeniser's relative encoding.
- Filtering removes this drift without interfering with the per-patch normalisation.

---

### HuBERT-ECG — Bandpass + Per-lead Z-score

HuBERT-ECG takes raw float tensors (12, 1000) and processes each lead independently
before mean-pooling to a 768-dimensional representation.

**Why bandpass + per-lead z-score:**
- HuBERT-ECG was pretrained on 9.1 M clinical ECGs acquired on hospital-grade
  machines, which apply hardware bandpass filtering (typically 0.05–150 Hz).
  Feeding raw signals with wander pushes the model out of its pretraining
  distribution.
- Per-lead z-score is needed because leads have naturally different voltage ranges
  (V5/V6 ~ 1–2 mV; V1 ~ 0.1–0.3 mV). Without normalisation the high-voltage leads
  dominate the mean-pooling step, effectively down-weighting diagnostically
  important low-amplitude leads. Per-lead z-score puts all 12 leads on equal
  footing before pooling.

In [ ]:
## 2. Data Loading and Preprocessing

Y = load_all_labels(DATA_PATH + 'ptbxl_database.csv', DATA_PATH + 'scp_statements.csv')

train_df = Y[Y.strat_fold < 9]
val_df   = Y[Y.strat_fold == 9]
test_df  = Y[Y.strat_fold == 10]

LEAD_IDX = 1  # Lead II

# ── Preprocessing primitives ──────────────────────────────────────────────────

def bandpass(sig, fs=100, low=0.5, high=40.0, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, sig).astype(np.float32)

def zscore(sig):
    std = sig.std()
    return ((sig - sig.mean()) / (std if std > 1e-8 else 1.0)).astype(np.float32)

def preprocess_1d(X, do_filter=True, normalise='none'):
    out = np.empty_like(X)
    for i, sig in enumerate(X):
        s = bandpass(sig) if do_filter else sig.copy()
        if normalise == 'zscore':
            s = zscore(s)
        out[i] = s
    return out

# ── Load raw Lead II arrays ───────────────────────────────────────────────────

def load_lead(df, lead_idx=LEAD_IDX):
    X, y = [], []
    for _, row in df.iterrows():
        sig, _ = wfdb.rdsamp(DATA_PATH + row['filename_lr'])
        X.append(sig[:, lead_idx].astype(np.float32))
        y.append(np.array(row['label_vec'], dtype=np.float32))
    return np.stack(X), np.stack(y)

print('Loading Lead II arrays (train / val / test)...')
X_train_raw, y_train = load_lead(train_df)
X_val_raw,   y_val   = load_lead(val_df)
X_test_raw,  y_test  = load_lead(test_df)
print(f'  Train : {X_train_raw.shape}')
print(f'  Val   : {X_val_raw.shape}')
print(f'  Test  : {X_test_raw.shape}')

# ── Apply model-specific preprocessing ────────────────────────────────────────

print('\nPreprocessing for HeartBERT (bandpass + z-score)...')
X_train_hb = preprocess_1d(X_train_raw, do_filter=True, normalise='zscore')
X_val_hb   = preprocess_1d(X_val_raw,   do_filter=True, normalise='zscore')
X_test_hb  = preprocess_1d(X_test_raw,  do_filter=True, normalise='zscore')

print('Preprocessing for ECG-PT (bandpass only)...')
X_train_ep = preprocess_1d(X_train_raw, do_filter=True, normalise='none')
X_val_ep   = preprocess_1d(X_val_raw,   do_filter=True, normalise='none')
X_test_ep  = preprocess_1d(X_test_raw,  do_filter=True, normalise='none')

# ── Filtered 12-lead dataset for HuBERT-ECG ──────────────────────────────────

class FilteredECGDataset(Dataset):
    """12-lead ECG dataset: bandpass filter + per-lead z-score applied on load."""
    def __init__(self, df, data_path):
        self.df        = df.reset_index(drop=True)
        self.data_path = data_path.rstrip('/')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        sig, _    = wfdb.rdsamp(f"{self.data_path}/{row['filename_lr']}")
        x         = sig.T.astype(np.float32)   # (12, 1000)
        for lead in range(x.shape[0]):
            s = bandpass(x[lead])
            x[lead] = zscore(s)
        y = torch.tensor(np.array(row['label_vec'], dtype=np.float32))
        return torch.tensor(x), y

train_ds_filt = FilteredECGDataset(train_df, DATA_PATH)
val_ds_filt   = FilteredECGDataset(val_df,   DATA_PATH)
test_ds_filt  = FilteredECGDataset(test_df,  DATA_PATH)
print('\nDatasets ready.')

## 3. Experiment 1 — HeartBERT LoRA r=8 (Bandpass + Z-score)

Fresh model loaded from Google Drive cache. LoRA adapters target `query` and
`value` projections (r=8, alpha=16). Trained on preprocessed Lead II signals.

In [ ]:
from src.models.heartbert import HeartBERTClassifier

model_hb = HeartBERTClassifier(num_labels=5)
model_hb.load()
model_hb.apply_peft(r=8, alpha=16, use_dora=False)

auc_hb, hist_hb = model_hb.fit(
    X_train_hb, y_train,
    X_val_hb,   y_val,
    experiment_name = '08_heartbert_lora_r8_filtered_zscore',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_peft'],
    save_dir        = RESULTS_PATH,
)
del model_hb
gc.collect()
torch.cuda.empty_cache()

## 4. Experiment 2 — ECG-PT LoRA r=8 (Bandpass only)

GPT-2 backbone (falls back from Tconnector/ecg-pt if unavailable). LoRA adapters
target `c_attn` (combined QKV). Trained on bandpass-filtered Lead II; patch
tokeniser handles per-patch normalisation internally.

In [ ]:
from src.models.ecgpt import ECGPTClassifier

model_ep = ECGPTClassifier(num_labels=5)
model_ep.load()
model_ep.apply_peft(r=8, alpha=16, use_dora=False)

auc_ep, hist_ep = model_ep.fit(
    X_train_ep, y_train,
    X_val_ep,   y_val,
    experiment_name = '08_ecgpt_lora_r8_filtered',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_peft'],
    save_dir        = RESULTS_PATH,
)
del model_ep
gc.collect()
torch.cuda.empty_cache()

## 5. Experiment 3 — HuBERT-ECG LoRA r=8 (Bandpass + Per-lead Z-score)

HuBERT-base (93 M params, pretrained on 9.1 M 12-lead ECGs). LoRA adapters
target `q_proj`, `k_proj`, `v_proj`. Signals preprocessed on-the-fly by
`FilteredECGDataset` defined in Cell 2.

In [ ]:
from src.models.hubert_ecg import HuBERTECGClassifier
from src.training.train_peft import run_peft_experiment

model_hub = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
model_hub.load()
model_hub.apply_peft(r=8, alpha=16, use_dora=False)
model_hub.to(device)

auc_hub, hist_hub, _ = run_peft_experiment(
    model_hub, train_ds_filt, val_ds_filt,
    experiment_name = '08_hubert_lora_r8_filtered_zscore',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_full'],
    save_dir        = RESULTS_PATH,
    num_workers     = CFG['training']['num_workers'],
)
del model_hub
gc.collect()
torch.cuda.empty_cache()

## 6. Test Set Evaluation

Evaluate all three models on the held-out test fold (fold 10) using their
respective preprocessed inputs. Results are compared against the r=8
raw-signal baselines from Notebook 05.

In [ ]:
from src.models.heartbert import HeartBERTClassifier
from src.evaluation.metrics import full_eval

EXP  = '08_heartbert_lora_r8_filtered_zscore'
prof = json.loads((Path(RESULTS_PATH) / EXP / 'profiling.json').read_text())

model = HeartBERTClassifier(num_labels=5)
model.load()
model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')

logits_hb  = model.predict_logits(X_test_hb)
results_hb = full_eval(logits_hb, y_test, run_bootstrap=True, n_bootstrap=1000)
results_hb['trainable_params']   = prof['trainable_params']
results_hb['checkpoint_size_mb'] = prof['checkpoint_size_mb']
results_hb['val_auc']            = auc_hb
print(f"HeartBERT filtered+zscore  AUC {results_hb['auc_macro']:.4f}  Fmax {results_hb['fmax']:.4f}")

del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from src.models.ecgpt import ECGPTClassifier

EXP  = '08_ecgpt_lora_r8_filtered'
prof = json.loads((Path(RESULTS_PATH) / EXP / 'profiling.json').read_text())

model = ECGPTClassifier(num_labels=5)
model.load()
model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')

logits_ep  = model.predict_logits(X_test_ep)
results_ep = full_eval(logits_ep, y_test, run_bootstrap=True, n_bootstrap=1000)
results_ep['trainable_params']   = prof['trainable_params']
results_ep['checkpoint_size_mb'] = prof['checkpoint_size_mb']
results_ep['val_auc']            = auc_ep
print(f"ECG-PT filtered-only       AUC {results_ep['auc_macro']:.4f}  Fmax {results_ep['fmax']:.4f}")

del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from src.models.hubert_ecg import HuBERTECGClassifier

EXP  = '08_hubert_lora_r8_filtered_zscore'
prof = json.loads((Path(RESULTS_PATH) / EXP / 'profiling.json').read_text())

model = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
model.load()
model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')
model.to(device)
model.eval()

test_loader = DataLoader(
    test_ds_filt, batch_size=1, shuffle=False,
    num_workers=CFG['training']['num_workers'],
)
logits_list, labels_list = [], []
with torch.no_grad():
    for x, y_b in test_loader:
        logits_list.append(model(x.to(device)).cpu())
        labels_list.append(y_b)

logits_hub  = torch.cat(logits_list).numpy()
labels_hub  = torch.cat(labels_list).numpy()
results_hub = full_eval(logits_hub, labels_hub, run_bootstrap=True, n_bootstrap=1000)
results_hub['trainable_params']   = prof['trainable_params']
results_hub['checkpoint_size_mb'] = prof['checkpoint_size_mb']
results_hub['val_auc']            = auc_hub
print(f"HuBERT-ECG filtered+zscore AUC {results_hub['auc_macro']:.4f}  Fmax {results_hub['fmax']:.4f}")

del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
## 7. Results Table

# ── Baseline results from Notebook 05 (raw input, r=8) ───────────────────────
BASELINES = [
    {'Model': 'HeartBERT',  'Preprocessing': 'Raw (no filter)',
     'AUC': 0.820, 'Fmax': 0.594, 'Trainable': 889_000, 'New': False},
    {'Model': 'ECG-PT',     'Preprocessing': 'Raw (no filter)',
     'AUC': 0.619, 'Fmax': 0.412, 'Trainable': 299_000, 'New': False},
    {'Model': 'HuBERT-ECG', 'Preprocessing': 'Raw (no filter)',
     'AUC': 0.549, 'Fmax': 0.399, 'Trainable': 295_000, 'New': False},
]

NEW_RESULTS = [
    {'Model': 'HeartBERT',  'Preprocessing': 'Bandpass + z-score',
     'AUC': results_hb['auc_macro'], 'Fmax': results_hb['fmax'],
     'Trainable': results_hb['trainable_params'], 'New': True},
    {'Model': 'ECG-PT',     'Preprocessing': 'Bandpass only',
     'AUC': results_ep['auc_macro'], 'Fmax': results_ep['fmax'],
     'Trainable': results_ep['trainable_params'], 'New': True},
    {'Model': 'HuBERT-ECG', 'Preprocessing': 'Bandpass + per-lead z-score',
     'AUC': results_hub['auc_macro'], 'Fmax': results_hub['fmax'],
     'Trainable': results_hub['trainable_params'], 'New': True},
]

# Add delta AUC vs baseline
baseline_auc = {'HeartBERT': 0.820, 'ECG-PT': 0.619, 'HuBERT-ECG': 0.549}
all_rows = []
for row in BASELINES + NEW_RESULTS:
    r = dict(row)
    if r['New']:
        r['Delta AUC'] = f"{r['AUC'] - baseline_auc[r['Model']]:+.4f}"
    else:
        r['Delta AUC'] = '—'
    all_rows.append(r)

df = pd.DataFrame(all_rows)[['Model','Preprocessing','AUC','Fmax','Trainable','Delta AUC']]
df['AUC']  = df['AUC'].apply(lambda x: f'{x:.4f}')
df['Fmax'] = df['Fmax'].apply(lambda x: f'{x:.4f}')
df['Trainable'] = df['Trainable'].apply(lambda x: f'{int(x):,}')

# Group with separator between baseline and new
print('\n=== Preprocessing Ablation Results (LoRA r=8, test fold 10) ===\n')
print(df.to_string(index=False))

# Save
out = {
    'baselines': BASELINES,
    'new_results': [
        {**r, 'auc_macro': r['AUC'], 'per_class_auc': res['per_class_auc'],
         'auprc_macro': res.get('auprc_macro', None), 'auc_ci_95': res.get('auc_ci_95','N/A')}
        for r, res in zip(NEW_RESULTS, [results_hb, results_ep, results_hub])
    ]
}
with open(OUT_DIR + 'results.json', 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f'\nSaved → {OUT_DIR}results.json')

## 8. Explainability on Best Model

Identify the model with the highest test AUC from the new preprocessing
experiments and run gradient saliency on a sample test record.
If HeartBERT wins, also visualise CLS attention weights over the letter tokens.

In [ ]:
from src.explainability.saliency import compute_saliency, top_salient_leads

LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']

# ── Identify best model ───────────────────────────────────────────────────────
scores = {
    'heartbert': results_hb['auc_macro'],
    'ecgpt':     results_ep['auc_macro'],
    'hubert':    results_hub['auc_macro'],
}
best_key = max(scores, key=scores.get)
print(f'Best model: {best_key.upper()}  (AUC {scores[best_key]:.4f})')

# ── Pick a test record ────────────────────────────────────────────────────────
# Use the MI example identified in notebook 01 (ecg_id=13815)
MI_EID = 13815
row = Y.loc[MI_EID]
sig_raw, meta = wfdb.rdsamp(DATA_PATH + row['filename_lr'])
true_class = [SUPERCLASSES[i] for i, v in enumerate(row['label_vec']) if v == 1]
print(f'Record   : ecg_id={MI_EID}  True class: {true_class}')

# ── Gradient saliency (works for all architectures) ───────────────────────────
if best_key == 'hubert':
    EXP = '08_hubert_lora_r8_filtered_zscore'
    model = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
    model.load()
    model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')
    model.to(device)
    model.eval()

    # Preprocess: bandpass + per-lead zscore
    x_12 = sig_raw.T.astype(np.float32)
    for lead in range(12):
        s = bandpass(x_12[lead])
        x_12[lead] = zscore(s)

    target_idx = SUPERCLASSES.index(true_class[0])
    saliency   = compute_saliency(model._head, x_12, target_idx, torch.device(device))
    top3       = top_salient_leads(saliency, LEAD_NAMES)
    probs      = torch.sigmoid(model(torch.tensor(x_12).unsqueeze(0).to(device))).detach().cpu().squeeze().numpy()

    print(f'Predicted probs: {dict(zip(SUPERCLASSES, probs.round(3)))}')
    print(f'Top-3 salient leads: {top3}')

    # Plot top-3 leads with saliency overlay
    fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
    t = np.arange(1000) / 100
    for ax, lead_name in zip(axes, top3):
        lead_idx = LEAD_NAMES.index(lead_name)
        sal_norm = saliency[lead_idx] / (saliency[lead_idx].max() + 1e-8)
        ax.plot(t, x_12[lead_idx], color='#1976d2', linewidth=0.9)
        ax.fill_between(t, x_12[lead_idx].min(), x_12[lead_idx].max(),
                        alpha=sal_norm * 0.5, color='#d32f2f')
        ax.set_ylabel(lead_name, rotation=0, ha='right', va='center')
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'Gradient saliency — HuBERT-ECG (filtered+zscore)  |  True: {true_class}', y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR + 'saliency_best_model.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUT_DIR}saliency_best_model.png')
    del model

elif best_key == 'heartbert':
    EXP = '08_heartbert_lora_r8_filtered_zscore'
    model = HeartBERTClassifier(num_labels=5)
    model.load()
    model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')

    lead_ii_raw = sig_raw[:, 1].astype(np.float32)
    lead_ii_pre = zscore(bandpass(lead_ii_raw))

    positions, attn = model.get_attention_weights(lead_ii_pre)
    probs = model.predict(lead_ii_pre[np.newaxis])

    print(f'Predicted probs: {dict(zip(SUPERCLASSES, probs[0].round(3)))}')

    fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
    t = np.arange(len(lead_ii_pre)) / 100
    axes[0].plot(t, lead_ii_pre, color='#1976d2', linewidth=0.9)
    axes[0].set_ylabel('Lead II (z-scored)')
    axes[0].grid(True, alpha=0.3)
    axes[1].fill_between(positions / 100, attn, color='#d32f2f', alpha=0.7)
    axes[1].set_ylabel('CLS attention')
    axes[1].set_xlabel('Time (s)')
    axes[1].grid(True, alpha=0.3)
    fig.suptitle(f'HeartBERT CLS attention  |  True: {true_class}', y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR + 'attention_best_model.png', dpi=150, bbox_inches='tight')
    plt.show()
    del model

else:  # ecgpt
    EXP = '08_ecgpt_lora_r8_filtered'
    model = ECGPTClassifier(num_labels=5)
    model.load()
    model.load_adapter(RESULTS_PATH + EXP + '/best_adapter')

    lead_ii_pre = bandpass(sig_raw[:, 1].astype(np.float32))
    probs = model.predict(lead_ii_pre[np.newaxis])
    print(f'Predicted probs: {dict(zip(SUPERCLASSES, probs[0].round(3)))}')
    print('Note: ECG-PT has no built-in attention visualisation; gradient saliency would require 12-lead reshape.')
    del model

gc.collect()
torch.cuda.empty_cache()
print('\nDone.')